# Modelo PD — Probability of Default
## Credit Risk IRB Platform | TribeTech 2026

Desenvolvimento e validação do modelo **PD (Probability of Default)** seguindo a abordagem IRB conforme Basel III e CRR.

---
**Referências Regulatórias:**
- Basel III — Artigos 143–191 CRR (Capital Requirements Regulation)
- EBA/GL/2017/06 — Directrizes sobre Definição de Incumprimento
- EBA/RTS/2016/03 — Modelos Internos — Requisitos de Validação
- EBA/GL/2021/07 — Backtesting e Planos de Correcção

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import duckdb
from sklearn.metrics import (
    roc_curve, auc, brier_score_loss,
    confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score,
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

from fastapi_ml.training.feature_engineering import (
    create_default_flag, engineer_features, PD_FEATURES,
)

# TribeTech Theme
TRB_RED    = '#E63C2F'
TRB_ORANGE = '#F97316'
TRB_DARK   = '#0F1117'
TRB_CARD   = '#1A1D27'
TRB_BORDER = '#2D3147'
TRB_TEXT   = '#E2E8F0'
TRB_MUTED  = '#94A3B8'

plt.rcParams.update({
    'figure.facecolor': TRB_DARK,
    'axes.facecolor':   TRB_CARD,
    'axes.edgecolor':   TRB_BORDER,
    'axes.labelcolor':  TRB_TEXT,
    'xtick.color':      TRB_MUTED,
    'ytick.color':      TRB_MUTED,
    'text.color':       TRB_TEXT,
    'grid.color':       TRB_BORDER,
    'grid.alpha':       0.5,
    'axes.grid':        True,
    'font.size':        11,
})

print('Ambiente carregado.')

## 1. Carregamento dos Dados e Modelos Treinados

In [ ]:
DATA_PATH = os.getenv(
    'DATA_PATH',
    '/home/alexmendes/bank_tech/Data/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv'
)

con = duckdb.connect()
df_raw = con.execute(f"""
    SELECT loan_status, loan_amnt, funded_amnt, term, int_rate, installment,
           grade, sub_grade, emp_length, home_ownership, annual_inc,
           verification_status, purpose, dti, delinq_2yrs, fico_range_low,
           fico_range_high, open_acc, pub_rec, revol_bal, revol_util,
           total_pymnt, recoveries, issue_d
    FROM read_csv_auto('{DATA_PATH}', ignore_errors=true)
    USING SAMPLE 300000 ROWS
""").df()
con.close()

df = create_default_flag(df_raw)
df = engineer_features(df)

# Filtro: apenas empréstimos com status final
final_statuses = {
    'Fully Paid','Charged Off','Default',
    'Does not meet the credit policy. Status:Charged Off',
    'Does not meet the credit policy. Status:Fully Paid',
    'Late (121-150 days)','Late (31-120 days)',
}
df = df[df['loan_status'].isin(final_statuses)].copy()

# Carregar modelos
with open('../fastapi_ml/artifacts/pd_model.pkl', 'rb') as f:
    xgb_model = pickle.load(f)
with open('../fastapi_ml/artifacts/pd_scorecard_lr.pkl', 'rb') as f:
    lr_model = pickle.load(f)

print(f'Dados: {len(df):,} observações')
print(f'Default rate: {df["is_default"].mean():.2%}')
print(f'Modelo PD: {type(xgb_model).__name__}')

## 2. Preparação dos Dados — Train/Test Split Temporal

In [ ]:
df_model = df[PD_FEATURES + ['is_default', 'grade', 'issue_d']].dropna(subset=PD_FEATURES)

# Split temporal 80/20
df_model = df_model.copy()
df_model['issue_d'] = pd.to_datetime(df_model['issue_d'], errors='coerce')

if df_model['issue_d'].notna().sum() > 1000:
    cutoff = df_model['issue_d'].quantile(0.80)
    train_mask = df_model['issue_d'] <= cutoff
    print(f'Split temporal: corte em {cutoff.date()}')
else:
    train_mask = np.random.RandomState(42).rand(len(df_model)) < 0.80
    print('Split aleatório 80/20 (issue_d não disponível)')

X_train = df_model[train_mask][PD_FEATURES].values
y_train = df_model[train_mask]['is_default'].values
X_test  = df_model[~train_mask][PD_FEATURES].values
y_test  = df_model[~train_mask]['is_default'].values

print(f'Treino:  {len(X_train):,} obs  | defaults: {y_train.sum():,} ({y_train.mean():.2%})')
print(f'Teste:   {len(X_test):,} obs  | defaults: {y_test.sum():,} ({y_test.mean():.2%})')

## 3. Comparação de Modelos: XGBoost vs Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# XGBoost — modelo carregado
y_xgb = xgb_model.predict_proba(X_test)[:, 1]

# LR — com standardização
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
lr_fresh  = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_fresh.fit(X_train_s, y_train)
y_lr = lr_fresh.predict_proba(X_test_s)[:, 1]

def compute_metrics(y_true, y_score, name=''):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    gini    = 2 * roc_auc - 1
    brier   = brier_score_loss(y_true, y_score)

    sorted_idx = np.argsort(y_score)
    y_s = y_true[sorted_idx]
    cum_bad  = np.cumsum(y_s) / y_s.sum()
    cum_good = np.cumsum(1-y_s) / (1-y_s).sum()
    ks = np.abs(cum_bad - cum_good).max()

    return {'modelo': name, 'gini': gini, 'ks': ks, 'auc': roc_auc, 'brier': brier}

results = pd.DataFrame([
    compute_metrics(y_test, y_xgb, 'XGBoost'),
    compute_metrics(y_test, y_lr,  'Logistic Regression'),
])

print('=== COMPARAÇÃO DE MODELOS PD ===')
print(results.to_string(index=False))
print()
winner = results.loc[results['gini'].idxmax(), 'modelo']
print(f'Modelo seleccionado: {winner} (maior Gini)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(TRB_DARK)

model_styles = [
    (y_xgb, 'XGBoost', TRB_RED),
    (y_lr,  'Logistic Regression', '#3B82F6'),
]

# Curvas ROC
ax = axes[0]
for y_score, name, color in model_styles:
    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={roc_auc:.3f})')
ax.plot([0,1],[0,1], color=TRB_MUTED, linestyle='--', lw=1)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('Curvas ROC — Comparação de Modelos PD', fontsize=11)
ax.legend(framealpha=0.2)

# Distribuição de Scores
ax = axes[1]
for y_score, name, color in model_styles:
    ax.hist(y_score[y_test==0], bins=50, alpha=0.4, color=color, label=f'{name} (Bom)')
    ax.hist(y_score[y_test==1], bins=50, alpha=0.6, color=color, label=f'{name} (Default)',
            histtype='step', linewidth=2)
ax.set_xlabel('Score PD')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição dos Scores PD por Classe', fontsize=11)
ax.legend(framealpha=0.2, fontsize=8)

plt.tight_layout()
plt.savefig('../fastapi_ml/artifacts/pd_model_comparison.png', dpi=150, bbox_inches='tight',
            facecolor=TRB_DARK)
plt.show()

## 4. Calibração do Modelo

A calibração verifica se as probabilidades estimadas correspondem às taxas de default observadas — requisito regulatório EBA para uso em cálculo de RWA.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(TRB_DARK)

for ax, (y_score, name, color) in zip(axes, model_styles):
    prob_true, prob_pred = calibration_curve(y_test, y_score, n_bins=10)

    ax.plot(prob_pred, prob_true, color=color, marker='o', lw=2, markersize=7,
            label=f'{name}')
    ax.plot([0,1],[0,1], color=TRB_MUTED, linestyle='--', lw=1.5, label='Calibração Perfeita')
    ax.fill_between(prob_pred, prob_true, prob_pred,
                    alpha=0.1, color=color, label='Desvio de Calibração')

    # ECE (Expected Calibration Error)
    n_bins_hist, bin_edges = np.histogram(y_score, bins=10)
    ece = np.mean(np.abs(prob_true - prob_pred))

    ax.set_xlabel('Probabilidade Prevista (Score PD)')
    ax.set_ylabel('Proporção Real de Defaults')
    ax.set_title(f'Diagrama de Calibração — {name}\nECE = {ece:.4f}', fontsize=11)
    ax.legend(framealpha=0.2)
    ax.set_xlim([0,1]); ax.set_ylim([0,1])

    ax.text(0.05, 0.90, f'ECE: {ece:.4f}\n{"✓ Boa calibração" if ece < 0.05 else "⚠ Verificar calibração"}',
            transform=ax.transAxes, fontsize=10, color=TRB_TEXT,
            bbox=dict(boxstyle='round', facecolor=TRB_BORDER, alpha=0.7))

plt.tight_layout()
plt.savefig('../fastapi_ml/artifacts/pd_calibration.png', dpi=150, bbox_inches='tight',
            facecolor=TRB_DARK)
plt.show()

## 5. Feature Importance (XGBoost SHAP-style)

Análise de importância de features — requisito de interpretabilidade EBA/ECB.

In [ ]:
# Feature importance do XGBoost
importance_gain  = xgb_model.get_booster().get_score(importance_type='gain')
importance_cover = xgb_model.get_booster().get_score(importance_type='cover')
importance_weight= xgb_model.get_booster().get_score(importance_type='weight')

imp_df = pd.DataFrame({
    'feature': PD_FEATURES,
    'gain':    [importance_gain.get(f'f{i}', 0) for i in range(len(PD_FEATURES))],
    'cover':   [importance_cover.get(f'f{i}', 0) for i in range(len(PD_FEATURES))],
    'weight':  [importance_weight.get(f'f{i}', 0) for i in range(len(PD_FEATURES))],
}).sort_values('gain', ascending=False)

# Normalizar
for col in ['gain','cover','weight']:
    imp_df[col] = imp_df[col] / imp_df[col].sum()

print('Feature Importance (Gain):')
print(imp_df[['feature','gain']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.patch.set_facecolor(TRB_DARK)
fig.suptitle('Feature Importance — XGBoost PD Model\n(Três Métricas de Importância)',
             fontsize=13, fontweight='bold')

for ax, metric, color, title in [
    (axes[0], 'gain',   TRB_RED,    'Gain (Redução Impureza)'),
    (axes[1], 'cover',  TRB_ORANGE, 'Cover (N Amostras)'),
    (axes[2], 'weight', '#3B82F6',  'Weight (N Splits)'),
]:
    df_sorted = imp_df.sort_values(metric)
    bars = ax.barh(df_sorted['feature'], df_sorted[metric], color=color,
                   alpha=0.85, edgecolor='none', height=0.65)
    for bar, val in zip(bars, df_sorted[metric]):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8, color=TRB_TEXT)
    ax.set_xlabel(f'{metric.capitalize()} (normalizado)')
    ax.set_title(title, fontsize=10, pad=8)

plt.tight_layout()
plt.savefig('../fastapi_ml/artifacts/pd_feature_importance.png', dpi=150, bbox_inches='tight',
            facecolor=TRB_DARK)
plt.show()

## 6. Backtesting — Validação Temporal

Análise do desempenho do modelo por coorte temporal — requisito EBA/GL/2021/07.

In [ ]:
df_bt = df_model.copy()
df_bt['issue_d'] = pd.to_datetime(df_bt['issue_d'], errors='coerce')
df_bt = df_bt.dropna(subset=['issue_d'] + PD_FEATURES)
df_bt['year'] = df_bt['issue_d'].dt.year

backtest_results = []
for yr in sorted(df_bt['year'].unique()):
    df_yr = df_bt[df_bt['year'] == yr]
    if len(df_yr) < 200 or df_yr['is_default'].sum() < 10:
        continue

    X_yr = df_yr[PD_FEATURES].values
    y_yr = df_yr['is_default'].values

    try:
        y_score_yr = xgb_model.predict_proba(X_yr)[:, 1]
        fpr, tpr, _ = roc_curve(y_yr, y_score_yr)
        roc_auc_yr  = auc(fpr, tpr)
        gini_yr     = 2 * roc_auc_yr - 1

        sorted_idx = np.argsort(y_score_yr)
        y_s = y_yr[sorted_idx]
        ks_yr = np.abs(
            np.cumsum(y_s)/y_s.sum() -
            np.cumsum(1-y_s)/(1-y_s).sum()
        ).max()

        backtest_results.append({
            'ano': yr,
            'n_obs': len(df_yr),
            'dr': y_yr.mean(),
            'gini': gini_yr,
            'ks': ks_yr,
            'auc': roc_auc_yr,
        })
    except Exception as e:
        print(f'  Erro {yr}: {e}')

bt_df = pd.DataFrame(backtest_results)
print('=== BACKTESTING TEMPORAL ===')
print(bt_df.to_string(index=False))

In [ ]:
if not bt_df.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.patch.set_facecolor(TRB_DARK)
    fig.suptitle('Backtesting Temporal — Modelo PD por Ano de Emissão\n(EBA/GL/2021/07)',
                 fontsize=13, fontweight='bold', y=1.01)

    years_str = bt_df['ano'].astype(str)

    # Gini
    ax = axes[0,0]
    ax.plot(years_str, bt_df['gini'], color=TRB_RED, marker='o', lw=2, markersize=7)
    ax.axhline(0.35, color=TRB_ORANGE, linestyle='--', alpha=0.7, label='Threshold EBA (0.35)')
    ax.axhline(0.20, color='#64748B', linestyle=':', alpha=0.7, label='Mínimo EBA (0.20)')
    ax.fill_between(years_str, bt_df['gini'], 0, alpha=0.1, color=TRB_RED)
    ax.set_title('Gini por Coorte', fontsize=11)
    ax.set_ylabel('Gini'); ax.legend(framealpha=0.2, fontsize=8)

    # KS
    ax = axes[0,1]
    ax.plot(years_str, bt_df['ks'], color=TRB_ORANGE, marker='s', lw=2, markersize=7)
    ax.axhline(0.25, color=TRB_RED, linestyle='--', alpha=0.7, label='Threshold EBA (0.25)')
    ax.fill_between(years_str, bt_df['ks'], 0, alpha=0.1, color=TRB_ORANGE)
    ax.set_title('KS por Coorte', fontsize=11)
    ax.set_ylabel('KS'); ax.legend(framealpha=0.2, fontsize=8)

    # Default Rate
    ax = axes[1,0]
    bars = ax.bar(years_str, bt_df['dr']*100, color='#3B82F6', alpha=0.8, edgecolor='none')
    for bar, val in zip(bars, bt_df['dr']*100):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                f'{val:.1f}%', ha='center', fontsize=9, color=TRB_TEXT)
    ax.set_title('Taxa de Default por Ano', fontsize=11)
    ax.set_ylabel('Taxa de Default (%)')

    # Volume
    ax = axes[1,1]
    bars = ax.bar(years_str, bt_df['n_obs'], color='#22C55E', alpha=0.7, edgecolor='none')
    for bar, val in zip(bars, bt_df['n_obs']):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
                f'{val:,}', ha='center', fontsize=9, color=TRB_TEXT)
    ax.set_title('Volume por Coorte (N obs)', fontsize=11)
    ax.set_ylabel('N Observações')

    for ax in axes.flat:
        ax.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig('../fastapi_ml/artifacts/pd_backtesting.png', dpi=150, bbox_inches='tight',
                facecolor=TRB_DARK)
    plt.show()

## 7. Análise por Grade — Discriminação Interna

In [ ]:
GRADE_COLORS = ['#22C55E','#86EFAC','#FDE68A','#FBBF24','#FB923C','#EF4444','#B91C1C']

df_grade = df_model.dropna(subset=PD_FEATURES + ['grade', 'is_default'])
X_all = df_grade[PD_FEATURES].values
y_all = df_grade['is_default'].values

try:
    scores_all = xgb_model.predict_proba(X_all)[:, 1]
except:
    scores_all = xgb_model.predict_proba(X_all)[:, 1]

df_grade = df_grade.copy()
df_grade['pd_score'] = scores_all

grade_stats = df_grade.groupby('grade').agg(
    n_obs     = ('is_default', 'count'),
    dr_obs    = ('is_default', 'mean'),
    pd_mean   = ('pd_score', 'mean'),
    pd_median = ('pd_score', 'median'),
).reset_index()

grade_stats['dr_obs']  = grade_stats['dr_obs'] * 100
grade_stats['pd_mean'] = grade_stats['pd_mean'] * 100

print('=== PD por Grade ===')
print(grade_stats.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(TRB_DARK)

grades = grade_stats['grade'].tolist()
gc = GRADE_COLORS[:len(grades)]

# Default rate observada vs PD prevista
ax = axes[0]
x = np.arange(len(grades))
w = 0.35
b1 = ax.bar(x - w/2, grade_stats['dr_obs'],  w, label='DR Observada',  color=TRB_RED,    alpha=0.85)
b2 = ax.bar(x + w/2, grade_stats['pd_mean'], w, label='PD Prevista (média)', color=TRB_ORANGE, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(grades)
ax.set_ylabel('Taxa (%)')
ax.set_title('DR Observada vs PD Prevista por Grade\n(Validação de Calibração IRB)', fontsize=11)
ax.legend(framealpha=0.2)

# Distribuição de scores por grade (boxplot)
ax = axes[1]
data_by_grade = [df_grade[df_grade['grade']==g]['pd_score'].values * 100
                 for g in grades]
bp = ax.boxplot(data_by_grade, labels=grades, patch_artist=True,
                medianprops=dict(color=TRB_TEXT, linewidth=2),
                whiskerprops=dict(color=TRB_MUTED),
                capprops=dict(color=TRB_MUTED),
                flierprops=dict(marker='.', markerfacecolor=TRB_MUTED, markersize=3, alpha=0.3))
for patch, color in zip(bp['boxes'], gc):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('Score PD (%)')
ax.set_title('Distribuição de Scores PD por Grade\n(Monotonia Esperada: A→G crescente)', fontsize=11)

plt.tight_layout()
plt.savefig('../fastapi_ml/artifacts/pd_by_grade.png', dpi=150, bbox_inches='tight',
            facecolor=TRB_DARK)
plt.show()

## 8. Relatório de Validação Regulatório — Sumário

In [ ]:
y_scores_test = xgb_model.predict_proba(X_test)[:, 1]
fpr_f, tpr_f, _ = roc_curve(y_test, y_scores_test)
roc_auc_f = auc(fpr_f, tpr_f)
gini_f    = 2 * roc_auc_f - 1
brier_f   = brier_score_loss(y_test, y_scores_test)

sorted_idx = np.argsort(y_scores_test)
y_s = y_test[sorted_idx]
ks_f = np.abs(
    np.cumsum(y_s)/y_s.sum() - np.cumsum(1-y_s)/(1-y_s).sum()
).max()

print()
print('╔══════════════════════════════════════════════════════════╗')
print('║    RELATÓRIO DE VALIDAÇÃO — MODELO PD IRB               ║')
print('║    TribeTech Credit Risk Platform | 2026                ║')
print('╠══════════════════════════════════════════════════════════╣')
print(f'║  Algoritmo  : XGBoost Gradient Boosting                ║')
print(f'║  Dataset    : Lending Club 2007–2018Q4 (amostra 300K)  ║')
print(f'║  Split      : Temporal 80/20                           ║')
print('╠══════════════════════════════════════════════════════════╣')
print(f'║  Gini       : {gini_f:.4f}  {"✓ APROVADO" if gini_f >= 0.35 else "⚠ VERIFICAR"}                   ║')
print(f'║  KS         : {ks_f:.4f}  {"✓ APROVADO" if ks_f >= 0.25 else "⚠ VERIFICAR"}                   ║')
print(f'║  AUC-ROC    : {roc_auc_f:.4f}  {"✓ APROVADO" if roc_auc_f >= 0.65 else "⚠ VERIFICAR"}                   ║')
print(f'║  Brier Score: {brier_f:.4f}  {"✓ APROVADO" if brier_f <= 0.25 else "⚠ VERIFICAR"}                   ║')
print('╠══════════════════════════════════════════════════════════╣')
print(f'║  Referência : EBA/GL/2017/06, Basel III Art. 143–191   ║')
print(f'║  Backtesting: EBA/GL/2021/07                           ║')
print('╚══════════════════════════════════════════════════════════╝')

# Guardar métricas
metrics_pd = {
    'gini': round(gini_f, 4), 'ks': round(ks_f, 4),
    'auc_roc': round(roc_auc_f, 4), 'brier': round(brier_f, 4),
    'model': 'XGBoost', 'n_test': len(y_test), 'dr_test': round(float(y_test.mean()), 4)
}
import json
with open('../fastapi_ml/artifacts/pd_metrics.json', 'w') as f:
    json.dump(metrics_pd, f, indent=2)
print()
print('Métricas guardadas em artifacts/pd_metrics.json')